# Assignment Introduction

This assignment involves simulating robot actions in a physics simulator. The main goal is to understand how the environment (including the robot, objects, and other elements) is represented within the simulator, and how low-level robot control can be implemented to execute tasks. You will be working with PyBullet, which is a lightweight yet powerful physics simulator widely used for robotics research. Although PyBullet supports a variety of robots and actions, for this assignment we will focus on the Franka Emika Robot.

In this assignment, you will work with a tabletop environment where a robotic manipulator arm can pick up objects and place them at desired locations. Humans perform such tasks easily by relying on vision, but in robotics this typically requires advanced machine learning methods, which we will explore later in the course. For now, we will use a simulator, which provides access to the ground-truth properties of objects such as their position, orientation, mass, and velocity. This makes it easier to focus on implementing the mechanics of pick-and-place operations without the added complexity of perception.

The assignment is structured as follows:

**Understanding PyBullet** -  Learn how PyBullet works, how to connect to it, and how to add objects to the environment.

**Objects in the Environment** - Study how objects and robots are represented in PyBullet, and how to access their geometry and dynamics.

**Robot Representation and Control** - Understand how the robot is represented in PyBullet and how to control it.

**Pick-and-Place Procedures** - Implement several pick-and-place operations to manipulate different objects in the tabletop and shelf environment.

The above objectives will improve your understanding of the simulator and how simulators work in general. The **assignment** shall then follow as:

**1.** *Perform a simple pick-place task in the table top environment with a cube object.*

<img src="simple_motion.gif" width="300">

**2.** *Utilise pick-place from previous task to perform the cleanup task.*

<img src="decluttering.gif" width="400">

**3.** *We move from table top setting to a shelf environment where you have to fetch an object (a bottle) which is behind another object (a cup).*

<img src="unshelfing.gif" width="350">

By completing this assignment, you will gain hands-on experience in working with a physics simulator, controlling a robot, and implementing basic manipulation strategies.

### Installation and Setup (one-time)

Pybullet is a light weight simulator and can be run in local laptops (windows, linux, mac) and workstation. You don't need CUDA compatible GPU. However, note that the laptop should have a graphics card to render the simulations. 

1. If you don't have conda, install using this [link](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html).
2. Create a conda environment using the yaml file.

``` conda env create -f pbullet.yml ```


Apart from following packages, pybullet also require OpenGL to be installed on your machine, it is generally system installed, but if you run into an issue you can keep OpenGL installation in mind

## 0. Getting Started
This section provides an example snippet showing how to import the PyBullet Python API, connect with the simulator, and add objects or a robot to the environment. Before proceeding with the assignment, you are encouraged to go through the PyBullet documentation to get an outline of its features and available functions.

In [2]:
# Importing required packages

import pybullet as p
import pybullet_data
import time, math
import numpy as np
import random
import os
from enum import Enum
from copy import deepcopy

pybullet build time: Jan 15 2026 18:31:44


### 0.1 Connecting to Physics client


In [3]:
#create a physics client
physics_client = p.connect(p.GUI) #P.GUI on computers with display or p.DIRECT for non-graphical version

print("The pybullet ships some useful data: ", pybullet_data.getDataPath())
p.setAdditionalSearchPath(pybullet_data.getDataPath()) #optionally set the search path for data files. 
p.setGravity(0,0,-9.8) #set the vector for gravity. This tells the gravity is acting in the negative z direction

The pybullet ships some useful data:  /home/aditya_ag00/College Courses/COL778/Assignments/venv/lib/python3.12/site-packages/pybullet_data
startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Mesa
GL_RENDERER=llvmpipe (LLVM 20.1.2, 256 bits)
GL_VERSION=4.5 (Core Profile) Mesa 25.0.7-0ubuntu0.24.04.2
GL_SHADING_LANGUAGE_VERSION=4.50
pthread_getconcurrency()=0
Version = 4.5 (Core Profile) Mesa 25.0.7-0ubuntu0.24.04.2
Vendor = Mesa
Renderer = llvmpipe (LLVM 20.1.2, 256 bits)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started


### 0.2 Spawning Objects
Most physics simulators, including PyBullet, require objects and robots to be represented in URDF (Unified Robot Description Format) or SDF (Simulation Description Format) in order to load them into the simulated environment. A URDF file is an XML-based format that describes the structure of a robot, including its links (rigid bodies), joints, geometry, visual appearance, and physical properties such as mass and inertia. In simpler terms, the URDF tells the simulator what the robot and the objects looks like and how it can move.

For this assignment, we will primarily use URDF files, as they are widely supported and relatively straightforward to work with. PyBullet provides built-in functions to load URDFs into the environment using commands like p.loadURDF(). Once loaded, these descriptions allow the simulator to handle the geometry and physics of the objects/robot automatically.

In [4]:
# First we add a ground plane to the environment. Each object in pybullet will be given a unique id.
planeId = p.loadURDF("plane.urdf")
print("The plane id is: ", planeId)

# Now you would be seeing a plane (blue and white grid) in the GUI window. 

ven = Mesa
The plane id is:  0
ven = Mesa


When adding an asset (such as a robot or an object) into the PyBullet environment, you need to specify its initial position and orientation in the 3D coordinate frame. The position is typically given as an (x, y, z) coordinate, which tells the simulator where in the world the object should be spawned.

For the orientation, PyBullet expects a quaternion representation. Quaternions are a mathematical way of representing 3D rotations, and while they are robust and widely used in robotics and computer graphics, they are not always intuitive to work with directly. To make things easier, PyBullet provides utility functions to convert from more intuitive Euler angles (roll, pitch, yaw) into quaternions. You may find it useful to specify the orientation in Euler angles first, and then convert them into the required quaternion format.

In [5]:
startPos = [0,0,1]
startOrientation = p.getQuaternionFromEuler([0,0,0])
print("The start position of r2d2 is: ", startPos)
print("The start orientation r2d2 is: ", startOrientation)
robotID = p.loadURDF("r2d2.urdf",startPos, startOrientation) #Loads a r2d2 robot at the specified position and orientation
# Now you would be seeing a r2d2 robot in the GUI window.

The start position of r2d2 is:  [0, 0, 1]
The start orientation r2d2 is:  (0.0, 0.0, 0.0, 1.0)


### 0.3 Simulating the environment

Note that when objects are first added to the PyBullet environment, they remain static and no physics is applied automatically. To simulate physics such as gravity, collisions, and motion, PyBullet provides stepSimulation() function.

The stepSimulation() function advances the simulation by a single time step. At each step, PyBullet updates the state of all objects according to the laws of physics, including forces, velocities, and interactions like collisions or contacts. By calling this function iteratively, you can observe the robot and objects move realistically within the environment.

In [6]:
p.resetBasePositionAndOrientation(robotID, startPos, startOrientation)

def simulate_world(num_steps_per_second):
    for i in range(num_steps_per_second):
        p.stepSimulation()
        time.sleep(1./num_steps_per_second) #This will make the simulation run at {num_steps}Hz. 
    #You can change this value to make the simulation run faster or slower.

simulate_world(100)
endPos, endOrn = p.getBasePositionAndOrientation(robotID)
print("pos after simulation: ",endPos)
print("orn after simulation: ",endOrn)


pos after simulation:  (3.1805784746229996e-08, 1.5092877444051277e-06, 0.4705357838700987)
orn after simulation:  (-8.851302156776823e-08, 2.3708517743198853e-08, 8.4690571719943e-08, 0.9999999999999922)


### 0.4 Disconnecting from the client

In [7]:
#once you are done, disconnect the physics client
p.disconnect() 

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


# 1. Pick and Place with a Manipulator
Now that you are familiar with the basic principles of the simulator, let us move on to creating a simple environment. In this section, we will set up a tabletop scenario with a manipulator arm and one or more objects. Once the environment is ready, we will implement a basic pick-and-place operation.

The goal here is to understand how to (1) define an environment, (2) position both the robot and objects within it, and (3) use low-level control to move the robot’s gripper so that it can grasp an object and place it at a desired location.

### 1.1 Setting up the environment

In [11]:
# Connect to physics client
physics_client = p.connect(p.GUI)
p.setGravity(0,0,-9.8) #set the vector for gravity. This tells the gravity is acting in the negative z direction

# Load ground plane
plane_position = [0,0,-1.1615/2]
planeId=p.loadURDF("plane.urdf",plane_position)

# Load a table (using pybullet_data only)
tablePosition = np.array([0.3,-0.1,0])+np.array(plane_position)
table_orientation = p.getQuaternionFromEuler([0, 0, -math.pi/2])
tableId=p.loadURDF("table/table.urdf",tablePosition,table_orientation)


startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=Mesa
GL_RENDERER=llvmpipe (LLVM 20.1.2, 256 bits)
GL_VERSION=4.5 (Core Profile) Mesa 25.0.7-0ubuntu0.24.04.2
GL_SHADING_LANGUAGE_VERSION=4.50
pthread_getconcurrency()=0
Version = 4.5 (Core Profile) Mesa 25.0.7-0ubuntu0.24.04.2
Vendor = Mesa
Renderer = llvmpipe (LLVM 20.1.2, 256 bits)
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = Mesa
ven = Mesa


In [10]:
#once you are done, disconnect the physics client
p.disconnect() 

numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
finished
numActiveThreads = 0
btShutDownExampleBrowser stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed


PyBullet allows you to configure parameters for the simulation like the step size etc. Read more about the parameters in the PyBullet documentation.

In [10]:
# frequency of steps
NUM_STEPS_PER_SECOND = 100

# set the step size in simulator using the parameter NUM_STEPS_PER_SECOND
def simulate_world(num_steps_per_second):
    for _ in range(num_steps_per_second):
        p.stepSimulation()
        time.sleep(1./240)

simulate_world(NUM_STEPS_PER_SECOND)

### 1.2 Understanding spawned objects
Now let us try to understand the objects we have spawned in the environment. For example, consider the table we just added. In PyBullet, every object (whether a robot or a simple asset like a table) can have multiple joints, links, and associated properties. Even though table have no actuators or control mechanisms, it is still represented in the simulator with a pose (position and orientation) and may have joints if defined in its URDF.

By querying an object’s properties, you can access:

 - Base pose: the position and orientation of the object in the world frame.

 - Joint information: if the asset has joints (e.g., in robots), you can get their type, limits, and current state.

 - Dynamics: such as mass, friction, and other physical parameters.

Understanding these properties is important because interacting with any object in the simulator, whether static like a table or dynamic like a robot arm, requires precise knowledge of its geometry, pose, and possible degrees of freedom.

Here we will investigate the spawned table object

<img src="table.png" width="200">

In [11]:
# Try to understand the table object
# get the number of joints in a object. 0 if the object doesn't have any joints.
num_table_joints = p.getNumJoints(tableId)

# get the body info of the object. It outputs name and the urdf file of the object.
table_body_info = p.getBodyInfo(tableId)

# # get the base position and orientation of the object
table_pose = p.getBasePositionAndOrientation(tableId) 

# # get the base velocity of the object.
table_vel = p.getBaseVelocity(tableId) 

# # get the base mass of the object. 
table_mass = p.getDynamicsInfo(tableId,-1)[0]


print('num joints in the table:', num_table_joints)
print('Body info:', table_body_info)
print('Base position and orientation of the table:', table_pose)
print('Base velocity of the table:', table_vel)
print('Mass of the table:', table_mass)

NameError: name 'tableId' is not defined

### 1.3 Adding a Simulated Robot Arm

In [9]:
# Spawn franka emika ROBOT robot using pybullet_data (at origin)
# TODO
robotId = p.loadURDF("franka_panda/panda.urdf",basePosition=[0,0,0],useFixedBase=True)

p.resetDebugVisualizerCamera(
    cameraDistance=1.2,
    cameraYaw=90,
    cameraPitch=-10,
    cameraTargetPosition=[0.5, 0, 0.5]
)

In [24]:
for i in range(p.getNumJoints(robotId)):
    info = p.getJointInfo(robotId, i)
    print(f"Index: {info[0]}, Name: {info[1].decode('utf-8')}, Type: {info[2]}")

Index: 0, Name: panda_joint1, Type: 0
Index: 1, Name: panda_joint2, Type: 0
Index: 2, Name: panda_joint3, Type: 0
Index: 3, Name: panda_joint4, Type: 0
Index: 4, Name: panda_joint5, Type: 0
Index: 5, Name: panda_joint6, Type: 0
Index: 6, Name: panda_joint7, Type: 0
Index: 7, Name: panda_joint8, Type: 4
Index: 8, Name: panda_hand_joint, Type: 4
Index: 9, Name: panda_finger_joint1, Type: 1
Index: 10, Name: panda_finger_joint2, Type: 1
Index: 11, Name: panda_grasptarget_hand, Type: 4


We have spawned a 7-Degree-of-Freedom (7-DoF) Franka Emika ROBOT robot. This means that the robot has 7 revolute joints, which define its primary arm motions. To better understand the robot model in the simulator, refer to the ```Controlling a Robot``` section in the PyBullet documentation. In this exercise, we will inspect the robot and check how many joints it contains, what types of joints they are, and what properties they have.

It is important to note that the URDF file of the robot usually defines more joints than what exists in the physical robot. This is because additional joints may be included for convenience in simulation, for example, fixed joints, mimic joints, or virtual joints to simplify control and collision handling.

In [25]:
## TODO: understand the joint types
    
# first we need to understand how many joints are there
ROBOT_NUM_JOINTS = p.getNumJoints(robotId)
print("num joints in ROBOT: ", ROBOT_NUM_JOINTS)

joint_map={p.JOINT_REVOLUTE:"REVOLUTE",p.JOINT_PRISMATIC:"PRISMATIC",p.JOINT_SPHERICAL:"SPHERICAL",p.JOINT_PLANAR:"PLANAR",p.JOINT_FIXED:"FIXED"}

# for each joint in the robot, print the joint name, joint_type, physical parameters: damping and friction, and joint limits
# print on seperate line for each of the joint and for a particular joint print the values with comma seperation  
for i in range(ROBOT_NUM_JOINTS):
    joint_info=p.getJointInfo(robotId,i)
    joint_name=joint_info[1].decode("utf-8")
    joint_type=joint_map.get(joint_info[2])
    damping=joint_info[6]
    lower_limit=joint_info[8]
    upper_limit=joint_info[9]
    print(f"{joint_name}, {joint_type}, {damping}, {lower_limit}, {upper_limit}")


num joints in ROBOT:  12
panda_joint1, REVOLUTE, 0.0, -2.9671, 2.9671
panda_joint2, REVOLUTE, 0.0, -1.8326, 1.8326
panda_joint3, REVOLUTE, 0.0, -2.9671, 2.9671
panda_joint4, REVOLUTE, 0.0, -3.1416, 0.0
panda_joint5, REVOLUTE, 0.0, -2.9671, 2.9671
panda_joint6, REVOLUTE, 0.0, -0.0873, 3.8223
panda_joint7, REVOLUTE, 0.0, -2.9671, 2.9671
panda_joint8, FIXED, 0.0, 0.0, -1.0
panda_hand_joint, FIXED, 0.0, 0.0, -1.0
panda_finger_joint1, PRISMATIC, 0.0, 0.0, 0.04
panda_finger_joint2, PRISMATIC, 0.0, 0.0, 0.04
panda_grasptarget_hand, FIXED, 0.0, 0.0, -1.0


### 1.4 Configure robot parameters
In many cases, we need to configure certain robot parameters before performing tasks. These parameters include the robot's **home position** (the default pose from which it starts), **the end-effector ID** (which identifies the gripper link), and the **lower and upper joint limits** that define the feasible range of motion for each joint. Setting these parameters correctly is important for safe and effective control of the robot in the simulator. You can also refer to examples provided by pybullet in their [repository](https://github.com/bulletphysics/bullet3/tree/master/examples/pybullet/gym/pybullet_robots)

Joint limits for the Franka Panda robot can be found in their [documentation](https://download.franka.de/documents/100010_Product%20Manual%20Franka%20Emika%20Robot_10.21_EN.pdf) (mind the units)

In [12]:
## TODO
# Store ids of moving joints (Revolute joints, prismatic joints[for fingers])
REVOLUTE_JOINTS = list(range(7))
PRISMATIC_JOINTS = [9,10]

# configure the robots end effector id, end effector is the part of the robot which will interact with the objects when commanded
ROBOT_END_EFFECTOR_INDEX = 11

# configuring home position for robot: in default configuration what shall be the states of each joints of the robot
ROBOT_HOME_POSITION = [0.002434193097806552, -0.7811217832664136, 0.009772560588829905, -2.062867921360752, -0.007282385107511599, 1.4879767515924243, 0.8049546850317025, 0.0] #joint angles in radians for the panda home position. These values can be obtained from the Moveit ROS or Franka Control Inteface.

# configure the joint limits
JOINT_LOWER_LIMITS = [-2.9671,-1.8326,-2.9671,-3.1416,-2.9671,-0.0873,-2.9671] # TODO
JOINT_UPPER_LIMITS = [2.9671,1.8326,2.9671,0.0,2.9671,3.8223,2.9671] # TODO


# The fingers of the robot are already configured for you. You don't need to do anything extra here. 
# finger parameters
ROBOT_FINGER_OPEN = 0.04
ROBOT_FINGER_CLOSED = 0
# creating a constraint to keep the fingers centered
def create_finger_constraint(p, robotId):
	finger_constraint = p.createConstraint(
				robotId, 9, robotId, 10,
				jointType=p.JOINT_GEAR,
				jointAxis=[1, 0, 0],
				parentFramePosition=[0, 0, 0],
				childFramePosition=[0, 0, 0])

	p.changeConstraint(finger_constraint, gearRatio=-1, erp=0.1, maxForce=50) #These values are taken from the pybullet examples https://github.com/bulletphysics/bullet3/blob/master/examples/pybullet/gym/pybullet_robots/panda/panda_sim_grasp.py
	return p, robotId

p, robotId = create_finger_constraint(p, robotId)

NameError: name 'robotId' is not defined

Great! Now that you understand the robot and its parameters, let’s start building code to interact with it. The first step is to determine whether the robot is currently moving. In PyBullet, this can be done by checking the velocity or momentum of each joint. If all joint velocities are close to zero, the robot is considered stationary; otherwise, it is in motion. Monitoring this is important because many control commands (such as planning a new trajectory) should only be executed when the robot is not already moving.

In [9]:
def is_robot_moving(p, robotid, joints):
    '''
    args
        p: pybullet_client
        robotid: id assigned to the spawned robot
        joints: the joint ids of the robot which can freely move
    returns
        bool (moving/steady)
    This function will help in identifying if our robot is in motion so we could bring it to rest before stopping the simulation
    '''
    # TODO
    for joint_id in joints:
        joint_state=p.getJointState(robotid,joint_id)
        if abs(joint_state[1])>1e-3:
            return True
    return False

### 1.5 Controlling the robot motions

Is your robot moving? It should not be, because we have not given it any motion commands yet. Let's now try to move it.

In PyBullet, a robot can be controlled in multiple ways, such as position control, velocity control, or torque control. For this assignment, we will use position control, as it is the simplest to implement and understand.

In position control, you move the robot by directly setting the desired angles for its joints. PyBullet then computes the required motion to reach those positions. By specifying different joint configurations, you can move the robot's links and ultimately position the end-effector at different locations in the workspace.


We recommend using the POSITION control, but you can explore more and learn about the PD controls [here](https://eng.libretexts.org/Bookshelves/Industrial_and_Systems_Engineering/Chemical_Process_Dynamics_and_Controls_(Woolf)/09%3A_Proportional-Integral-Derivative_(PID)_Control/9.02%3A_P_I_D_PI_PD_and_PID_control#:~:text=Proportional%2DDerivative%20(PD)%20Control). This will also improve your understanding of parameters such as damping in control.

In [13]:
def move_robot_to_position(p, robotid, target_positions:list):
    '''
    args
        p: pybullet_client
        robotID: namespace id given to the robot
        target_positions: the target values for each joint
    returns
        It moves the robot to the mentioned target position and should return 
        true if succeeded and false if failed
    this function will be used to set joint commands on the robot to guide it's joints to the target_positions
    '''
    # TODO
    p.setJointMotorControlArray(robotid,jointIndices=REVOLUTE_JOINTS[:len(target_positions)],
                                controlMode=p.POSITION_CONTROL,
                                targetPositions=target_positions)
    simulate_world(240)
    tolerance=1e-2
    for i,joint_idx in enumerate(REVOLUTE_JOINTS[:len(target_positions)]):
        joint_state=p.getJointState(robotid,joint_idx)
        if abs(joint_state[0]-target_positions[i])>tolerance:
            return False
    return True

In [184]:
target_angles = [0.2,0.2,1,-0.8,1,0.9,1] 
move_robot_to_position(p,robotId,target_angles)

True

You can utilise this function to move robot to a defined default position. With provided values the homing action shall look like


<img src="home_position.png" width="300">

Opening or closing the gripper works in a very similar way to moving any other joint. The main difference is that instead of controlling the arm joints, you set the target positions for the gripper joints. By specifying small target values, you can make the fingers come closer together (closing), and by setting larger values, you can make them move apart (opening). 

In [14]:
def command_fingers(p, robotid, open:bool):
    '''
    args
        p: pybullet_client
        robotID: namespace id given to the robot
        open: do we want the robot to open its fingers, otherwise close them
    returns
        bool (success/failure)

    This function will simply close or open robots gripper, so to grasp or release the object
    '''
    # TODO
    if open:
        target_pos=ROBOT_FINGER_OPEN
    else:
        target_pos=ROBOT_FINGER_CLOSED

    for joint in [9, 10]:
        p.setJointMotorControl2(robotid, joint, 
                                controlMode=p.POSITION_CONTROL, 
                                targetPosition=target_pos,
                                force=200)
    simulate_world(240)

    tolerance=1e-2
    joint_state=p.getJointState(robotid,9)
    if abs(joint_state[0]-target_pos)>tolerance:
        return False
    return True

In [316]:
command_fingers(p,robotId,open=False)

False

### 1.6 Manipulating the objects

#### Picking up the object

Spawn the cube object

In [323]:
# using pybullet_data
# TODO 
cube=p.loadURDF("cube_small.urdf",basePosition=[0.5,0.0,0.1],globalScaling=1.5)

In [14]:
print(p.getBasePositionAndOrientation(cube))

((0.5, 0.0, 0.1), (0.0, 0.0, 0.0, 1.0))


Now that we can control both the robot arm and the gripper, let's perform a pick operation on the cube. The process can be broken down into the following steps:

1. **Get the cube's pose**: Query the simulator to obtain the position and orientation of the cube. This gives us the target location where the gripper must move.

2. **Compute joint angles using inverse kinematics**: The robot cannot be directly commanded to move its end-effector to an (x, y, z) position. Instead, we need to compute the corresponding joint angles that will bring the gripper to the cube's position. This is done using inverse kinematics (IK).

3. **Move the robot arm**: Using position control, send the computed joint angles to the robot so that the arm moves the gripper to the cube's location.

4. **Close the gripper**: Once the gripper is aligned with the cube, apply the same control method to the gripper joints to close the fingers and grasp the object.

💡 Hint: You can re-use the helper functions that you have already implemented for controlling joints, reading object poses, and moving the robot.

In [15]:
def pick_object(p, robotid, objectid):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        objectid: namespace id given to the object
    returns
        bool (success/failure)

    This function shall command physics client to pick objectid with robotid
    '''
    ## HINTs
    # First use the objectid to get its position and orientation,
    # Use pybullets inverse kinematics to get robots target joint positions corresponding to the object pose
    # TODO
    command_fingers(p,robotid,open=True)
    
    obj_pos,_=p.getBasePositionAndOrientation(objectid)
    target_pos=obj_pos # (x,y,z)
    joint_angles=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                              targetPosition=target_pos,
                                              upperLimits=JOINT_UPPER_LIMITS,
                                              lowerLimits=JOINT_LOWER_LIMITS,
                                              restPoses=ROBOT_HOME_POSITION[:7])

    # print(joint_angles[:7])
    
    if not move_robot_to_position(p,robotid,joint_angles[:len(REVOLUTE_JOINTS)-1]):
        return False,None
    
    command_fingers(p,robotid,open=False)  

    # we have to create a constraint between fingers and object as it is slipping sometimes
    # we can either increase the force on the fingers or create a fixed constraint
    # I choose creating constraint as increasing force is causing some problems

    grasp_constraint = p.createConstraint(
        parentBodyUniqueId=robotid,
        parentLinkIndex=ROBOT_END_EFFECTOR_INDEX,
        childBodyUniqueId=objectid,
        childLinkIndex=-1,
        jointType=p.JOINT_FIXED,
        jointAxis=[0,0,0],
        parentFramePosition=[0,0,0],
        childFramePosition=[0,0,0]
    )

    # target_pos_above=[obj_pos[0],obj_pos[1],obj_pos[2]+0.1] # slightly above the object
    # joint_angles_above=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
    #                                           targetPosition=target_pos_above,
    #                                           upperLimits=JOINT_UPPER_LIMITS,
    #                                           lowerLimits=JOINT_LOWER_LIMITS,
    #                                           restPoses=ROBOT_HOME_POSITION[:7])

    # if not move_robot_to_position(p,robotid,joint_angles_above[:7]):
    #     return False

    # obj_pos,_=p.getBasePositionAndOrientation(objectid)
    # if obj_pos[2]<0.1:  
    #     print("Object not lifted")  
    #     return False
    return True,None
    

In [347]:
# p.removeBody(robotId)
move_robot_to_position(p,robotId,ROBOT_HOME_POSITION[:7])
p.resetBasePositionAndOrientation(cube,[0.5, 0.0, 0.1], [0.0, 0.0, 0.0, 1.0])

In [296]:
p.resetDebugVisualizerCamera(
    cameraDistance=0.9,
    cameraYaw=0,
    cameraPitch=10,
    cameraTargetPosition=[0.5, 0, 0.5]
)

#### Place the object

Similarly place the object at the desired position

In [16]:
# I added an objectid parameter also to check whether the object drop location and target pose match
def place_at_target_position(p, robotid, target_pose,objectid,constraint):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        target_pose: location at which the object in robots hand is to be placed
    returns
        bool (success/failure)

    this function shall command the physics cliet to place the object at the target_pose
    '''
    # TODO
    target_orn=p.getQuaternionFromEuler([0,-math.pi,0])
    # ! For cube use this z offset for placing 
    # target_pos=[target_pose[0],target_pose[1],target_pose[2]+0.037]
    # target_pos=target_pose
    # ! For fruits use this z offset for placing 
    # target_pos=[target_pose[0],target_pose[1],target_pose[2]+0.05]
    joint_angles=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                              targetPosition=target_pose,
                                              upperLimits=JOINT_UPPER_LIMITS,
                                              lowerLimits=JOINT_LOWER_LIMITS,
                                              restPoses=ROBOT_HOME_POSITION[:7])

    if not move_robot_to_position(p,robotid,joint_angles[:len(REVOLUTE_JOINTS)-1]):
        return False
    
    # print("Removing constraint:", constraint)
    # print("All constraints:", p.getNumConstraints())

    if constraint is not None:
        p.removeConstraint(constraint)
    # print("All constraints:", p.getNumConstraints())

    if not command_fingers(p,robotid,open=True):
        return False
    
    simulate_world(240)

    obj_pos,_=p.getBasePositionAndOrientation(objectid)
    distance=np.linalg.norm(np.array(obj_pos)-np.array(target_pose))
    if distance>0.1:  
        print("Object was not lifted or dropped midway")  
        return False

    return True

    # ! for part 1- dont use constraint, target offset=0.03, use finger force=200
    # ! for part 2- use constraint, target offset=0.05, use finger force=100

In [345]:
success,grasp_constraint=pick_object(p,robotId,cube)
# p.removeConstraint(grasp_constraint)
print(success)
place_at_target_position(p,robotId,[0.7,0.0,0.1],cube,grasp_constraint)
# p.removeBody(cube)

True
b3Printf: b3Warning[examples/SharedMemory/PhysicsClientSharedMemory.cpp,808]:

b3Printf: removeConstraint failed


True

### 1.7 Pick and place with crane like operations

Now try placing the objects at different positions like the left or right side of the table. Is the placement always stable? You might notice that sometimes the robot collides with the cube either while picking or while placing. This happens because the robot has 7 degrees of freedom, which means there can be multiple possible joint configurations (solutions) that reach the same end-effector position. Some of these solutions may cause unwanted collisions.

To avoid this, it is often desirable to define **waypoints** for the motion rather than moving directly to the target. This is similar to how a crane operates: it does not swing the load directly to the final position but instead follows a safe sequence of motions.

Here, we will implement crane-like grasping:

1. First, move the robot above the object at a safe height.

2. Then, slowly descend vertically until the gripper reaches the object.

3. After grasping, lift the object upward to clear the surroundings.

4. Move the robot back to the home position while holding the object.

5. For placing, follow the reverse sequence: move above the target location, descend, release the object, and finally return the robot to home.

This waypoint-based approach provides safer and more reliable pick-and-place operations, reducing the chances of collisions and ensuring more stable placements. 

Here is an example of how it can work with the robot

<img src="waypoints_motion.gif" width="200">

In [17]:
def crane_pick_object(p, robotid, objectid):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        objectid: namespace id given to the object
    returns
        bool (success/failure)

    There are many ways a robot can reach target location and sometimes this may lead to undesirable behaviors
    for this purpose, we define an approximate path (through `waypoints`) so robot doesn't take absurd paths
    '''
    # TODO
    command_fingers(p,robotid,open=True)

    obj_pos,_=p.getBasePositionAndOrientation(objectid)
    obj_pos_above=[obj_pos[0],obj_pos[1],obj_pos[2]+0.15]
    target_orn=p.getQuaternionFromEuler([0,-math.pi,0])
    joint_angles_above=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                              targetPosition=obj_pos_above,
                                            #   targetOrientation=target_orn,
                                              upperLimits=JOINT_UPPER_LIMITS,
                                              lowerLimits=JOINT_LOWER_LIMITS,
                                              restPoses=ROBOT_HOME_POSITION[:7])
    move_robot_to_position(p,robotid,joint_angles_above[:len(REVOLUTE_JOINTS)-1]) # now robot has reached above object

    success,grasp_constraint=pick_object(p,robotid,objectid) # we then pick the object from above
    if not success:
        return False,None

    move_robot_to_position(p,robotid,joint_angles_above[:len(REVOLUTE_JOINTS)-1]) # now robot has reached above object after picking

    move_robot_to_position(p,robotid,ROBOT_HOME_POSITION[:len(REVOLUTE_JOINTS)-1]) # now robot has reached at home position

    return True,grasp_constraint


def crane_place_object(p, robotid, target_pose,objectid,constraint):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        target_pose: location at which the object in robots hand is to be placed
    returns
        bool (success/failure)
    '''
    # TODO
    target_pos_above=[target_pose[0],target_pose[1],target_pose[2]+0.15]
    # target_orn=p.getQuaternionFromEuler([0,-math.pi,0])
    joint_angles_above=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                                    targetPosition=target_pos_above,
                                                    # targetOrientation=target_orn,
                                                    upperLimits=JOINT_UPPER_LIMITS,
                                                    lowerLimits=JOINT_LOWER_LIMITS,
                                                    restPoses=ROBOT_HOME_POSITION[:7])
    move_robot_to_position(p,robotid,joint_angles_above[:len(REVOLUTE_JOINTS)-1]) # now robot has reached above object

    place_at_target_position(p,robotid,target_pose,objectid,constraint) # we then place the object from above

    move_robot_to_position(p,robotid,joint_angles_above[:len(REVOLUTE_JOINTS)-1])

    move_robot_to_position(p,robotid,ROBOT_HOME_POSITION[:len(REVOLUTE_JOINTS)-1]) # now robot has reached at home position

### 1.8 Perform a pick and place operation 
1. with normal pick and place functions and 
2. with crane like pick and place functioning

try to analyze the difference in motion paths with both methods

In [341]:
# TODO
move_robot_to_position(p,robotId,ROBOT_HOME_POSITION[:7])
p.resetBasePositionAndOrientation(cube,[0.5, 0.0, 0.1], [0.0, 0.0, 0.0, 1.0])

# normal pick place
success,grasp_constraint=pick_object(p,robotId,cube)
place_at_target_position(p,robotId,[0.7,0.0,0.1],cube,grasp_constraint)

move_robot_to_position(p,robotId,ROBOT_HOME_POSITION[:7])
p.resetBasePositionAndOrientation(cube,[0.5, 0.0, 0.1], [0.0, 0.0, 0.0, 1.0])

# crane like pick place
success,grasp_constraint=crane_pick_object(p,robotId,cube)
crane_place_object(p,robotId,[0.7,0.0,0.1],cube,grasp_constraint)

b3Printf: b3Warning[examples/SharedMemory/PhysicsClientSharedMemory.cpp,808]:

b3Printf: removeConstraint failed
b3Printf: b3Warning[examples/SharedMemory/PhysicsClientSharedMemory.cpp,808]:

b3Printf: removeConstraint failed


#### 1.9 Remove the cube from the scene
Remove the manipulable object to prepare the environment for the next task, while keeping the ground plane, table at the specified locations, and the robot.

In [348]:
# TODO
p.removeBody(cube) 




# 2. Collection and Cleanup
In this part of the assignment, you are provided with a household environment containing several fruits lying on the table. Your task, as the programmer of a helping robot, is to tidy the area by collecting and placing the fruits onto the plate positioned beside the robot. The plate is on a small shelf, providing an elevation. 

You may reuse your pick-and-place implementation from the previous part to execute the action sequences.

💡 Hint: Begin by verifying the target location of the plate. Conduct several iterations of placing fruits onto the plate, ensuring that collisions with other objects are avoided.


#### 2.1 The Environment

In [ ]:
p.disconnect()

In [ ]:
def table_environment(p):
    '''
    args
        p: provide the name given to the pybullet client
    returns
        objId's of the objects of interest in the scene
    '''
    # Connect to physics client
    # physics_client = p.connect(p.GUI)
    p.setGravity(0,0,-9.8) #set the vector for gravity. This tells the gravity is acting in the negative z direction

    base_position = [0,0,-1.1615/2]
    planeId = p.loadURDF("plane.urdf", basePosition = base_position)

    tablePosition = np.array([0.3,-0.1,0])+np.array(base_position)
    table_orientation = p.getQuaternionFromEuler([0, 0, -math.pi/2])
    tableId = p.loadURDF("table/table.urdf", tablePosition, table_orientation, globalScaling=1)
    block_position = [.1, .5, .05]
    sId = p.loadURDF('assets/Collection/small_shelf.urdf', block_position, globalScaling=.6)

    plate_position = [.1, .5, .31]
    plateId = p.loadURDF('assets/Collection/plate.urdf', plate_position, globalScaling=.1)

    lemon_position, orange_position, peach_position, plum_position = [0.48,-0.1,0.07], [0.38,-0.16,0.07], [0.52,-0.19,0.07], [0.44,-0.26,0.08]

    lemonId = p.loadURDF("assets/Collection/lemon.urdf", lemon_position, globalScaling=.06)
    orangeId = p.loadURDF("assets/Collection/orange.urdf", orange_position, globalScaling=.06)
    peachId = p.loadURDF("assets/Collection/peach.urdf", peach_position, globalScaling=.06)
    plumId = p.loadURDF("assets/Collection/plum.urdf", plum_position, globalScaling=.06)

    simulate_world(600)

    return plateId, orangeId, plumId, lemonId, peachId

plateId, orangeId, plumId, lemonId, peachId = table_environment(p)

In [99]:
import numpy as np

def get_object_diameter(p, objectId):
    aabb_min, aabb_max = p.getAABB(objectId)

    size = np.array(aabb_max) - np.array(aabb_min)
    diameter = np.max(size)

    return diameter

fruits = {
    "orange": orangeId,
    "plum": plumId,
    "lemon": lemonId,
    "peach": peachId
}

for name, fid in fruits.items():
    r = get_object_diameter(p, fid)
    print(f"{name} diameter ≈ {r:.3f} m")

print(f"plate diameter ≈ {get_object_diameter(p, plateId):.3f} m")

orange diameter ≈ 0.084 m
plum diameter ≈ 0.071 m
lemon diameter ≈ 0.054 m
peach diameter ≈ 0.049 m
plate diameter ≈ 0.270 m


In [ ]:
def clear_table_environment(p, object_ids: list):
    '''
    args:
        p: the pybullet client
        object_ids: a list or tuple containing the IDs of objects to remove
    returns:
        None
    '''
    for obj_id in object_ids:
        # p.removeBody removes the object from the simulation by its ID
        p.removeBody(obj_id)
    
    # Optional: step the simulation to update the visual state
    simulate_world(1) 
    print(f"Successfully removed {len(object_ids)} objects.")

# Usage:
# Capture the IDs from your environment function
ids_to_remove = [planeId,tableId,plateId, orangeId, plumId, lemonId, peachId]

# Remove them
clear_table_environment(p, ids_to_remove)

The environment should spawn objects like:

<img src="declutter_env.png" width="200">

#### 2.2 Perform the Cleanup Task

In [271]:
ee_pos = p.getLinkState(robotId, ROBOT_END_EFFECTOR_INDEX)[0]

p.resetDebugVisualizerCamera(
    cameraDistance=1.2,
    cameraYaw=90,
    cameraPitch=-5,
    cameraTargetPosition=ee_pos
)

In [54]:
robotId = p.loadURDF("franka_panda/panda.urdf",basePosition=[0,0,0],useFixedBase=True)

In [258]:
p.resetBasePositionAndOrientation(orangeId,[0.38,-0.16,0.07],[0.0,0.0,0.0,1.0])
p.resetBasePositionAndOrientation(plumId,[0.44,-0.26,0.08],[0.0,0.0,0.0,1.0])
p.resetBasePositionAndOrientation(lemonId,[0.48,-0.1,0.07],[0.0,0.0,0.0,1.0])
p.resetBasePositionAndOrientation(peachId,[0.52,-0.19,0.07],[0.0,0.0,0.0,1.0])
p.resetBasePositionAndOrientation(plateId,[.1, .5, .31],[0.0,0.0,0.0,1.0])
move_robot_to_position(p,robotId,ROBOT_HOME_POSITION[:7])

True

In [259]:
# For each object configure target location and perform pick-place
# TODO
plate_pos,_=p.getBasePositionAndOrientation(plateId)
# angles=[math.pi/4,3*math.pi/4,5*math.pi/4,-math.pi/4]
angles=[0,math.pi/2,math.pi,-math.pi/2]

for i,obj_id in enumerate([orangeId,plumId, lemonId, peachId]):
    success,grasp_constraint=crane_pick_object(p,robotId,obj_id)

    target_pos=(plate_pos[0]+0.03*math.cos(angles[i]),plate_pos[1]+0.03*math.sin(angles[i]),plate_pos[2])
    crane_place_object(p,robotId,target_pos,obj_id,grasp_constraint)

    print(target_pos)
    # print(p.getBasePositionAndOrientation(obj_id)[0])

print(p.getBasePositionAndOrientation(orangeId)[0])
print(p.getBasePositionAndOrientation(plumId)[0])
print(p.getBasePositionAndOrientation(lemonId)[0])
print(p.getBasePositionAndOrientation(peachId)[0])

(0.12997700886018906, 0.500009544133591, 0.3035584023843698)
(0.09997700886018906, 0.530009544133591, 0.3035584023843698)
(0.06997700886018907, 0.500009544133591, 0.3035584023843698)
(0.09997700886018906, 0.470009544133591, 0.3035584023843698)
(0.15928618472006684, 0.5305868206766938, 0.31822241240309085)
(0.11048337179056071, 0.5656486586416791, 0.31198350995634627)
(0.057168056151155285, 0.5031502736247286, 0.31296554270666166)
(0.09140330291466427, 0.4782979233452552, 0.31362197659655494)


In [50]:
print("Removing constraint:", grasp_constraint)
print("All constraints:", p.getNumConstraints())
p.removeConstraint(grasp_constraint)
print("Constraints after removal:", p.getNumConstraints())


Removing constraint: 5
All constraints: 3
Constraints after removal: 3
b3Printf: b3Warning[examples/SharedMemory/PhysicsClientSharedMemory.cpp,808]:

b3Printf: removeConstraint failed


In [47]:
p.removeConstraint(grasp_constraint)

b3Printf: b3Warning[examples/SharedMemory/PhysicsClientSharedMemory.cpp,808]:

b3Printf: removeConstraint failed


# 3. Lateral Grasping
In this last part of the assignment, you will work with a shelf environment where objects are placed on the shelves. Note that the crane-like grasping approach used earlier may not work here due to the limited overhead clearance. To safely pick the objects, you need to use a **lateral grasping** strategy, that is, move the gripper sideways into the shelf, align it with the object, and then close the fingers to grasp before retracting.

#### 3.1 The Environment
The environment is setup for you with two objects, a cup in front of a bottle. Your task here is to make agent pick the bottle from the shelf and place it on the table.

In [ ]:
p.disconnect()

In [29]:
def lateral_grasping_environment(p):
    '''
    args
        p: provide the name given to the pybullet client
    returns
        objId's: ids of objects of interest

    this function will steup the lateral grasping environment 
    '''
    # disconnect any physics client running already
    if p.isConnected():
        p.disconnect()

    import os
    # connect to the physics server
    physics_client = p.connect(p.GUI)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.configureDebugVisualizer(p.COV_ENABLE_SHADOWS, 1)
    p.setGravity(0,0,-9.8)

    base_position = [0,0,-1.1615/2]
    planeId = p.loadURDF("plane.urdf", basePosition = base_position)

    #Load a table
    tablePosition = np.array([0.35,0.1,0])+np.array(base_position) 
    # these values are aligned so robot could be placed at the origin
    table_orientation = p.getQuaternionFromEuler([0, 0, -math.pi/2])
    tableId = p.loadURDF("table/table.urdf", tablePosition, table_orientation, globalScaling=1)

    panda_position = [0,0,0]
    panda_orientation = p.getQuaternionFromEuler([0, 0, 0])
    robotId = p.loadURDF("franka_panda/panda.urdf", panda_position, panda_orientation, useFixedBase = 1)
        
    p.setTimeStep(1/NUM_STEPS_PER_SECOND)
    # p.setRealTimeSimulation(USE_REAL_TIME)

    p, robotId = create_finger_constraint(p, robotId)

    p.setAdditionalSearchPath(os.path.join(os.getcwd(), 'assets'))

    shelfPos = np.array([0.3,-0.8,.7177])+np.array(base_position)
    shelfOrientation = p.getQuaternionFromEuler([-math.pi/2-0.01,0,0])
    shelfId = p.loadURDF('Shelf/rack/urdf/rack.urdf', shelfPos, shelfOrientation, globalScaling=1)

    cupId = p.loadURDF('Shelf/cup_0/cup_small.urdf', [0.32,-0.69,1.08-.58], p.getQuaternionFromEuler([0,0,0]), globalScaling=0.9)

    bottleId = p.loadURDF('Shelf/plastic_water_bottle/urdf/plastic_water_bottle.urdf', [0.21,-0.86,1.053-.58], p.getQuaternionFromEuler([-1.57,0,0]), globalScaling=0.7)

    return bottleId, cupId, robotId

bottleId, cupId, robotId = lateral_grasping_environment(p)
simulate_world(500)

The environment setup for this task shall look similar to 

<img src="shelf_env.png" width="200">

### 3.2 Manipulating objects in the Shelf

In [30]:
ee_pos = p.getLinkState(robotId, ROBOT_END_EFFECTOR_INDEX)[0]

p.resetDebugVisualizerCamera(
    cameraDistance=0.5,      # pull back a bit
    cameraYaw=145,           # side view (diagonal)
    cameraPitch=-10,         # angled from above
    cameraTargetPosition=[
        ee_pos[0],
        ee_pos[1],
        ee_pos[2] - 0.1      # slightly below EE → shows table/shelf
    ]
)

In [22]:
move_robot_to_position(p,robotId,ROBOT_HOME_POSITION[:7])

True

In [18]:
# ! only 5 initial angles are moving and rest(6,7) are fixed

In [ ]:
def lateral_pick_object(p, robotid, objectid):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        objectid: namespace id given to the object
    returns
        bool (success/failure)
        
    picking an object requires it to grasp in a specific way, so you may have to configure grasping for each type of object
    '''
    # TODO    

Grasping objects in general is a difficult task since objects can have variety of geometries. This assignment focusses on manipulator motions than grasping, hence we don't expect for you to implement a generalized grasping solution. 

💡 Hint: 1. The URDF files spawned here present their own challenges (such as orientations), so you may need to implement specific grasping solutions for each object. First, identify how the simulator has set up the objects in its Cartesian frame. Then, use the object geometry information available from the simulator to successfully pick up the object while avoiding any collisions.

2. You can use the function p.getAABB(objectId,-1) to obtain the 3D axis-aligned bounding box (AABB) of the object, which is the smallest cuboid aligned with the world axes that fully encloses the object.

3. For this task you can analyse to the motion followed by agent in follwing references


In [36]:
# aabbMin,aabbMax=p.getAABB(cupId,-1)
# height=aabbMax[2]-aabbMin[2]
# width=aabbMax[0]-aabbMin[0]
# length=aabbMax[1]-aabbMin[1]
# print("bottle height:",height)
# print("bottle width:",width)
# print("bottle length:",length)
# print(aabbMin)
# print(aabbMax)
p.getBasePositionAndOrientation(cupId)

((0.28520840246738627, -0.6463061493894282, 0.5133731224030523),
 (-0.02175736993572128,
  -0.012717900086534105,
  0.006512572630639401,
  0.9996611717320023))

In [64]:
def grasp_bottle(p, robotid, bottleid):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot3
        bottleid: namespace id given to the bottle object
    returns
        bool (success/failure)
    '''
    # TODO
    aabbMin,aabbMax=p.getAABB(bottleId,-1)
    center_height=(aabbMax[2]+aabbMin[2])/2
    center_width=(aabbMax[0]+aabbMin[0])/2
    center_length=(aabbMax[1]+aabbMin[1])/2
    target_pos=[center_length,center_width,center_height]
    obj_pos,_=p.getBasePositionAndOrientation(bottleid)
    joint_angles=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                              targetPosition=target_pos,
                                              lowerLimits=JOINT_LOWER_LIMITS,
                                              upperLimits=JOINT_UPPER_LIMITS,
                                              restPoses=ROBOT_HOME_POSITION[:7])


def grasp_cup(p, robotid, cupid):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        cupid: namespace id given to the cup object
    returns
        bool (success/failure)
    '''
    # TODO
    # aabbMin,aabbMax=p.getAABB(cupid,-1)
    # center_height=(aabbMax[2]+aabbMin[2])/2
    # center_width=(aabbMax[0]+aabbMin[0])/2
    # center_length=(aabbMax[1]+aabbMin[1])/2
    # target_pos=[center_length,center_width,center_height]
    command_fingers(p,robotid,open=True)

    obj_pos,_=p.getBasePositionAndOrientation(cupid)
    target_pos=[obj_pos[0],obj_pos[1]+0.5,obj_pos[2]]
    joint_angles=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                              targetPosition=target_pos,
                                              lowerLimits=JOINT_LOWER_LIMITS,
                                              upperLimits=JOINT_UPPER_LIMITS,
                                              restPoses=ROBOT_HOME_POSITION[:7])
    move_robot_to_position(p,robotid,joint_angles[:len(REVOLUTE_JOINTS)-1])

    target_orn=p.getQuaternionFromEuler([0, -math.pi/2, math.pi/2])
    target_angles=p.calculateInverseKinematics(robotid,ROBOT_END_EFFECTOR_INDEX,
                                              targetPosition=obj_pos,
                                              targetOrientation=target_orn,
                                              lowerLimits=JOINT_LOWER_LIMITS,
                                              upperLimits=JOINT_UPPER_LIMITS,
                                              restPoses=ROBOT_HOME_POSITION[:7])
    move_robot_to_position(p,robotid,target_angles[:len(REVOLUTE_JOINTS)])

In [ ]:
# frequency of steps
NUM_STEPS_PER_SECOND = 100

# set the step size in simulator using the parameter NUM_STEPS_PER_SECOND
def simulate_world(num_steps_per_second):
    for _ in range(num_steps_per_second):
        p.stepSimulation()
        time.sleep(1./240)

simulate_world(NUM_STEPS_PER_SECOND)

In [65]:
move_robot_to_position(p,robotId,ROBOT_HOME_POSITION[:7])
grasp_cup(p,robotId,cupId)

In [367]:
def lateral_place_object(p, robotid, objectid):
    '''
    args
        p: pybullet_client
        robotid: namespace id given to the robot
        objectid: namespace id given to the object
    returns
        bool (success/failure)
    since the object is grasped laterally placing will also have to follow similar motions
    '''
    # TODO

### 3.3 Performing the task
Pick the bottle from the shelf and place it on the table

In [368]:
# TODO